# Frequency-Aware Latent Graph Diffusion (FALD) — Colab Workspace

**Run this notebook in a GPU runtime** (Runtime > Change runtime type > T4/A100).

This notebook is the single entry point for running the entire project on Google Colab.
It clones the repo, installs all dependencies (DiGress, PyG, ORCA, graph-tool),
and provides runnable cells for every stage from the `PLAN.md`.

### Current status (as of the last Windows session)

| Stage | Status |
|---|---|
| 1. Environment & DiGress | ✅ |
| 2. Evaluation harness | ✅ |
| 3. Spectral utilities & SignNet | ✅ |
| 3.5. Adjacency diffusion baseline | ✅ |
| 4. Autoencoder investigation | ✅ (decision: adjacency path) |
| 5. Unconditional adjacency diffusion | ✅ |
| 6. Kill gate (oracle conditioning) | ✅ PASSED |
| 6.5. Discrete DiGress validity repair | ⚠ red on validity |
| 7–10 | Not started |

**What this notebook provides:** everything needed to continue from Stage 6.5 onward,
plus the ability to re-run any earlier stage to verify results on Colab hardware.

---
## 0. Environment Setup

Run these cells once per Colab session. They are idempotent.

In [1]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

Tesla T4, 15360 MiB, 580.82.07


In [2]:
import os

REPO_URL = "https://github.com/TamaraBluzer/Frequency_aware_diffusion-.git"
REPO_DIR = "/content/FALD"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only || echo "Pull failed (maybe local changes); using existing checkout."

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into '/content/FALD'...
remote: Enumerating objects: 240, done.
remote: Counting objects: 100% (240/240), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 240 (delta 106), reused 183 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (240/240), 5.55 MiB | 11.71 MiB/s, done.
Resolving deltas: 100% (106/106), done.
Working directory: /content/FALD


In [3]:
# Create .pth file so DiGress's 'src' package is importable in subprocesses
import site
pth_path = site.getsitepackages()[0] + '/digress-src.pth'
with open(pth_path, 'w') as f:
    f.write('/content/FALD/third_party/digress')
    print(f'Wrote {pth_path}')
    # Verify it works
    import importlib, sys
    if 'src' in sys.modules: del sys.modules['src']
    site.addsitedir(site.getsitepackages()[0])
    from src.models.transformer_model import XEyTransformerLayer
    print(f'SUCCESS: XEyTransformerLayer imported from {XEyTransformerLayer.__module__}')

Wrote /usr/local/lib/python3.13/dist-packages/digress-src.pth


ModuleNotFoundError: No module named 'src.models'

In [ ]:
# Test that subprocess Python can import DiGress's src.models (the .pth fix)
!python -c "from src.models.transformer_model import XEyTransformerLayer; print('SUCCESS:', XEyTransformerLayer)"

In [ ]:
!rm -rf third_party/digress

In [ ]:
# Verify imports work
import torch
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}, GPU: {torch.cuda.get_device_name(0)}")

import networkx as nx
import numpy as np
from fald.data import load_splits, build_condition_tensors
from fald.eval import GraphEvaluator
from fald.eval.orca import is_available as orca_available
from fald.models import AdjacencyDiffusion, AdjacencyDiffusionConfig, DiscreteAdjacencyDiffusion
from fald.paths import repo_root, work_dir, data_dir, third_party_dir, checkpoints_dir, results_dir

print(f"ORCA available: {orca_available()}")
print(f"Repo root: {repo_root()}")
print(f"Work dir: {work_dir()}")
print("All imports OK.")

### Optional: Mount Google Drive for persistent checkpoints

Colab VMs are ephemeral — files are lost when the runtime disconnects.
Mount Google Drive to persist checkpoints and results across sessions.

In [4]:
# Run the full setup: DiGress clone+patch, ORCA build, all pip installs.
# Takes ~2-3 minutes on first run, <30s on re-run.
!bash scripts/colab_setup.sh

== FALD Colab setup ==
-- GPU detected --
Tesla T4, 15360 MiB, 580.82.07
-- nvcc reports CUDA 12.8 --
-- Using PyG wheel tag: cu124 --
-- Cloning DiGress at pinned commit --
Cloning into '/content/FALD/third_party/digress'...
remote: Enumerating objects: 593, done.
remote: Counting objects: 100% (396/396), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 593 (delta 322), reused 226 (delta 226), pack-reused 197 (from 1)
Receiving objects: 100% (593/593), 4.58 MiB | 8.75 MiB/s, done.
Resolving deltas: 100% (381/381), done.
Note: switching to '780242b8d3e7d78316bb5cf90c639fb0cd4c6079'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-na

In [ ]:
# Uncomment these lines to mount Google Drive and use it for heavy artifacts.
# Once mounted, FALD_WORK_DIR redirects checkpoints/data/third_party there.

# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_WORK_DIR = "/content/drive/MyDrive/FALD"
# os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
# os.environ["FALD_WORK_DIR"] = DRIVE_WORK_DIR
# print(f"FALD_WORK_DIR set to {DRIVE_WORK_DIR}")
# print("Checkpoints and data will persist across Colab sessions.")

---
## 1. Stage 1 — DiGress Smoke Test

Already passed. Re-run to verify Colab environment works end-to-end.

In [ ]:
# Quick ConGress smoke test (5 epochs, tiny model — not a real training run)
%cd {os.path.join(third_party_dir(), 'digress', 'src')}
!MPLBACKEND=Agg python main.py dataset=planar model=continuous general.name=stage1_smoke \
    general.wandb=disabled general.gpus=1 \
    train.batch_size=8 train.n_epochs=2 \
    general.number_chain_steps=10 model.diffusion_steps=50 model.n_layers=2
%cd {REPO_DIR}

---
## 2. Stage 2 — Evaluation Harness Calibration

Verifies that the MMD evaluator is working correctly: train-vs-train near zero,
train-vs-ER large.

In [ ]:
!python scripts/calibrate_eval.py --dataset planar

---
## 3. Stage 3 — Spectral Sanity Checks

Verifies eigendecomposition, band selection, SignNet invariance, and the component-count offset.

In [ ]:
!python scripts/spectral_sanity.py

---
## 4. Stage 5 — Unconditional Adjacency Diffusion (`none` baseline)

Trains the continuous adjacency diffusion model with no spectral condition.
This is the baseline that Stage 6's conditioned models must beat.

In [ ]:
# Full training run — ~15-30 min on T4 depending on batch size.
# Reduce --epochs for a quick check; the model was trained at 200 epochs.
!python scripts/train_adjacency_diffusion.py \
    --dataset planar \
    --band none \
    --k 8 \
    --epochs 200 \
    --batch-size 16 \
    --seed 0

In [ ]:
# Seeds 1 and 2 for error bars
!python scripts/train_adjacency_diffusion.py --dataset planar --band none --k 8 --epochs 200 --seed 1
!python scripts/train_adjacency_diffusion.py --dataset planar --band none --k 8 --epochs 200 --seed 2

---
## 5. Stage 6 — Kill Gate: Oracle Spectral Conditioning

The central hypothesis test. `low, k=8` with oracle spectra must beat `none`.

**Previous result (Windows):** Ratio: `low` 157.97 ± 16.20 vs `none` 327.23 ± 8.80.
Kill gate PASSED.

In [ ]:
# Train low-band conditioned model (3 seeds)
for seed in [0, 1, 2]:
    !python scripts/train_adjacency_diffusion.py \
        --dataset planar --band low --k 8 --epochs 200 --seed {seed}

In [ ]:
# Train high-band and random-band controls (3 seeds each)
for band in ["high", "random"]:
    for seed in [0, 1, 2]:
        !python scripts/train_adjacency_diffusion.py \
            --dataset planar --band {band} --k 8 --epochs 200 --seed {seed}

In [ ]:
# Condition sensitivity check (T9 from PLAN.md)
!python scripts/check_condition_sensitivity.py

In [ ]:
# Summarize the pilot results with bootstrap CIs
!python scripts/summarize_adjacency_pilot.py

---
## 6. Stage 6.5 — Discrete Diffusion Validity Repair

The continuous model has 0% Planar validity (consistent with published ConGress).
Try discrete D3PM to recover structural validity while keeping the conditioning advantage.

**Previous result:** 0% validity after 500 epochs — needs more capacity/steps to match
published DiGress (10 layers, 1000 steps, all auxiliary features).

In [10]:
# Discrete diffusion — larger model to attempt DiGress-level validity.
# This is the active frontier. Adjust n-layers, timesteps, and epochs as needed.
!PYTHONPATH=/content/FALD/third_party/digress:$PYTHONPATH python scripts/train_adjacency_diffusion.py \
    --dataset planar \
    --process discrete \
    --band none \
    --k 8 \
    --epochs 500 \
    --n-layers 6 \
    --timesteps 200 \
    --cycle-features \
    --allow-gate-failure \
    --seed 0

planar: train=128 val=32 process=discrete band=none k=8
model=2.74M timesteps=200 positive_weight=1.00 (class-balance value would be 10.34) amp=True
epoch    1 train=0.41277 val=0.32176
epoch   10 train=0.19363 val=0.20152
epoch   20 train=0.19548 val=0.20664
epoch   30 train=0.19588 val=0.20657
epoch   40 train=0.18503 val=0.18245
epoch   50 train=0.18790 val=0.22476
epoch   60 train=0.18635 val=0.18032
epoch   70 train=0.18816 val=0.18404
epoch   80 train=0.18165 val=0.19117
epoch   90 train=0.18607 val=0.17234
epoch  100 train=0.19595 val=0.17536
epoch  110 train=0.16440 val=0.18242
epoch  120 train=0.18773 val=0.18217
epoch  130 train=0.19124 val=0.18703
epoch  140 train=0.19116 val=0.23620
epoch  150 train=0.19177 val=0.20321
epoch  160 train=0.19178 val=0.16814
epoch  170 train=0.18550 val=0.16361
epoch  180 train=0.19429 val=0.17967
epoch  190 train=0.20996 val=0.16982
epoch  200 train=0.19246 val=0.19962
epoch  210 train=0.18629 val=0.18411
epoch  220 train=0.18682 val=0.19301


In [11]:
# Discrete with low-band conditioning
!python scripts/train_adjacency_diffusion.py \
    --dataset planar \
    --process discrete \
    --band low \
    --k 8 \
    --epochs 500 \
    --n-layers 6 \
    --timesteps 200 \
    --cycle-features \
    --allow-gate-failure \
    --seed 0

planar: train=128 val=32 process=discrete band=low k=8
model=2.74M timesteps=200 positive_weight=1.00 (class-balance value would be 10.34) amp=True
epoch    1 train=0.40382 val=0.29472
epoch   10 train=0.09793 val=0.09263
epoch   20 train=0.09421 val=0.08749
epoch   30 train=0.09512 val=0.08411
epoch   40 train=0.08577 val=0.07489
epoch   50 train=0.09580 val=0.08981
epoch   60 train=0.09087 val=0.07268
epoch   70 train=0.08296 val=0.07651
epoch   80 train=0.08902 val=0.07975
epoch   90 train=0.09295 val=0.07182
epoch  100 train=0.09304 val=0.07466
epoch  110 train=0.07711 val=0.07578
epoch  120 train=0.08909 val=0.07512
epoch  130 train=0.08869 val=0.07440
epoch  140 train=0.08059 val=0.08315
epoch  150 train=0.06813 val=0.06509
epoch  160 train=0.07172 val=0.05496
epoch  170 train=0.05825 val=0.04611
epoch  180 train=0.06258 val=0.04923
epoch  190 train=0.06259 val=0.04879
epoch  200 train=0.06079 val=0.05323
epoch  210 train=0.06175 val=0.04620
epoch  220 train=0.06100 val=0.04756
e

In [6]:
# Diagnose src import issue
!ls /content/FALD/third_party/digress/src/
!ls /content/FALD/third_party/digress/src/models/
!python -c "import sys; sys.path.insert(0, '/content/FALD/third_party/digress'); import src; print(src.__file__); from src.models.transformer_model import XEyTransformerLayer; print('OK')"

analysis  diffusion		       diffusion_model.py  main.py  models
datasets  diffusion_model_discrete.py  __init__.py	   metrics  utils.py
__init__.py  layers.py	transformer_model.py
/content/FALD/third_party/digress/src/__init__.py
OK


### Scaling up toward published DiGress

Published DiGress uses 10 layers, 1000 timesteps, and all auxiliary features.
Colab T4 (16GB) can handle this; A100 is faster.

In [8]:
# Write patch script to /tmp and run it
!cat > /tmp/patch_import.py << 'PYEOF'
import pathlib, re
f = pathlib.Path('/content/FALD/fald/models/adjacency_diffusion.py')
t = f.read_text()
# Replace the _import_digress_layer function
pattern = r'def _import_digress_layer\(\):.*?return XEyTransformerLayer'
new = """def _import_digress_layer():
    import importlib.util, sys
        from ..paths import third_party_dir
            target = third_party_dir() / "digress" / "src" / "models" / "transformer_model.py"
                if target.is_file():
                        spec = importlib.util.spec_from_file_location(
                                    "_digress_transformer_model", str(target),
                                                submodule_search_locations=[],
                                                        )
                                                                mod = importlib.util.module_from_spec(spec)
                                                                        sys.modules[spec.name] = mod
                                                                                spec.loader.exec_module(mod)
                                                                                        return mod.XEyTransformerLayer
                                                                                            raise ImportError(
                                                                                                    "Could not import DiGress XEyTransformerLayer from " + str(target)
                                                                                                        )"""
                                                                                                        t2 = re.sub(pattern, new, t, flags=re.DOTALL)
                                                                                                        assert t2 != t, "Pattern not found!"
                                                                                                        f.write_text(t2)
                                                                                                        print("Patched successfully")
                                                                                                        PYEOF
                                                                                                        !python /tmp/patch_import.py

IndentationError: unexpected indent (3777216709.py, line 24)

In [6]:
# Attempt closer to published DiGress scale
!python scripts/train_adjacency_diffusion.py \
    --dataset planar \
    --process discrete \
    --band low \
    --k 8 \
    --epochs 1000 \
    --n-layers 10 \
    --timesteps 500 \
    --cycle-features \
    --allow-gate-failure \
    --seed 0

planar: train=128 val=32 process=discrete band=low k=8
model=4.56M timesteps=500 positive_weight=1.00 (class-balance value would be 10.34) amp=True
epoch    1 train=0.38797 val=0.26959
epoch   10 train=0.10204 val=0.09329
epoch   20 train=0.10385 val=0.07975
epoch   30 train=0.09449 val=0.08928
epoch   40 train=0.09361 val=0.07457
epoch   50 train=0.09425 val=0.08222
epoch   60 train=0.08009 val=0.07688
epoch   70 train=0.08221 val=0.07117
epoch   80 train=0.09174 val=0.06940
epoch   90 train=0.09826 val=0.07270
epoch  100 train=0.08159 val=0.07757
epoch  110 train=0.08874 val=0.07124
epoch  120 train=0.09239 val=0.07202
epoch  130 train=0.08198 val=0.07548
epoch  140 train=0.07262 val=0.06212
epoch  150 train=0.06982 val=0.05818
epoch  160 train=0.07014 val=0.04451
epoch  170 train=0.06105 val=0.04644
epoch  180 train=0.05578 val=0.04220
epoch  190 train=0.04945 val=0.04752
epoch  200 train=0.04782 val=0.04343
epoch  210 train=0.05085 val=0.03053
epoch  220 train=0.04900 val=0.02998
e

---
## 7. Stage 7 — The Frequency Sweep (Headline Result)

Six arms × k ∈ {2, 4, 8, 16, 32} on Planar, 3 seeds each.
Still oracle-conditioned so the frequency question is isolated.

**Note:** Run this on the process type that has non-zero validity,
or run it on continuous to report Ratio (with the validity caveat noted).

In [ ]:
# Full frequency sweep — continuous model (reports Ratio, not V.U.N.)
# This is a large grid; consider running overnight on Colab Pro.
import itertools

bands = ["none", "low", "high", "random"]
ks = [2, 4, 8, 16, 32]
seeds = [0, 1, 2]

for band, k, seed in itertools.product(bands, ks, seeds):
    if band == "none" and k != 8:
        continue  # none is the same regardless of k
    print(f"\n{'='*60}")
    print(f"  band={band}  k={k}  seed={seed}")
    print(f"{'='*60}")
    !python scripts/train_adjacency_diffusion.py \
        --dataset planar --band {band} --k {k} --epochs 200 --seed {seed}

In [ ]:
# Reduced SBM grid: low, high, none at k ∈ {2, 8, 32}
sbm_bands = ["none", "low", "high"]
sbm_ks = [2, 8, 32]

for band, k, seed in itertools.product(sbm_bands, sbm_ks, seeds):
    if band == "none" and k != 8:
        continue
    print(f"\n{'='*60}")
    print(f"  SBM: band={band}  k={k}  seed={seed}")
    print(f"{'='*60}")
    !python scripts/train_adjacency_diffusion.py \
        --dataset sbm --band {band} --k {k} --epochs 200 --seed {seed}

---
## 8. Stage 8 — Learned Spectral Prior

Small diffusion model over `(n, λ_k, U_k)` so we can sample end-to-end
without oracle inputs. This stage is not yet implemented — add cells here
as the model is built.

In [ ]:
# TODO: Stage 8 implementation
print("Stage 8 not yet implemented. See PLAN.md for specification.")

---
## 9. Stage 9 — Downstream Experiment & Controls

- SBM community-count classification augmentation study
- Ablations: cluster-conditioning, Laplacian variant, SignNet
- Memorization audit

In [ ]:
# TODO: Stage 9 implementation
print("Stage 9 not yet implemented. See PLAN.md for specification.")

---
## 10. Stage 10 — Report Figures

Regenerate all figures from the `results/` directory.

In [ ]:
# TODO: Figure generation script
print("Stage 10 not yet implemented. See PLAN.md for specification.")

---
## Utilities

### Run tests

In [ ]:
!python -m pytest tests/ -v --tb=short

### Inspect results

In [ ]:
import json
from pathlib import Path

results = sorted(Path("results").glob("*.json"))
print(f"Found {len(results)} result files:\n")
for path in results:
    with open(path) as f:
        data = json.load(f)
    band = data.get("band", "?")
    k = data.get("k", "?")
    seed = data.get("seed", "?")
    ratio = data.get("evaluation", {}).get("ratio", "N/A") if data.get("evaluation") else "N/A"
    gate = data.get("gate_passed", "N/A")
    print(f"  {path.name:55s}  band={band:6s} k={k:>2} seed={seed}  Ratio={ratio!s:>8s}  gate={gate}")

### Save results to Google Drive

In [5]:
# Uncomment to copy results to Drive before the runtime disconnects.
# Make sure you mounted Drive in section 0 above.

# import shutil
# drive_results = "/content/drive/MyDrive/FALD/results"
# os.makedirs(drive_results, exist_ok=True)
# for f in Path("results").glob("*.json"):
#     shutil.copy2(f, drive_results)
#     print(f"Copied {f.name}")
# print(f"\nResults saved to {drive_results}")

### Commit and push results back to GitHub

In [ ]:
# Uncomment and fill in your token to push results from Colab.
# Use a fine-grained personal access token with Contents:write scope.

# GITHUB_TOKEN = "ghp_..."  # paste your token here
# !git remote set-url origin https://{GITHUB_TOKEN}@github.com/TamaraBluzer/Frequency_aware_diffusion-.git
# !git add results/*.json results/figures/
# !git commit -m "Add Colab results" || echo "Nothing to commit."
# !git push